# First Look: Winnipeg Mosquito Trap Counts

City of Winnipeg Open Data, *Daily Adult Mosquito Trap Data* (https://data.winnipeg.ca/d/du7c-8488), exported
2026-09-13. I filtered the export to complete seasons, 2015 to 2025: the 2026 season is still being counted.

In [1]:
import pandas as pd

traps = pd.read_csv("data/raw/traps-2015-2025.csv")
traps.shape

(880, 44)

In [2]:
traps.head()

,Count Date,City Wide Daily Average,Trap Days,North West 1,North West 2,North West 3,North West 4,North West 5,North West 6,North West 7,...,South West Average,Rural AA,Rural BB,Rural CC,Rural DD,Rural EE,Rural FF,Rural GG,Rural HH,Rural II
0,04/05/2015,1,3,1,2,1,0,2,0,1,...,1,1,0,0,1,0,3,0,2,7
1,06/05/2015,1,2,2,1,2,1,3,0,3,...,1,2,5,0,1,0,3,2,3,4
2,08/05/2015,0,2,0,0,0,0,0,0,1,...,0,0,0,0,0,0,1,1,1,0
3,11/05/2015,0,3,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,1
4,13/05/2015,0,2,1,0,0,1,1,0,1,...,0,2,0,0,1,0,0,0,0,1


**What one row is:** one count day. Checked: no date appears twice.

In [3]:
traps["Count Date"].duplicated().sum()

0

## Types and missing values

In [4]:
print("missing cells:", traps.isna().sum().sum())
traps.dtypes.value_counts()

missing cells: 0


object    38
int64      6
Name: count, dtype: int64

Nothing is missing, but 37 of the trap columns are text. Why?

In [5]:
trap_cols = [c for c in traps.columns if traps[c].dtype == "object" and c != "Count Date"]
values = traps[trap_cols].stack()
values[~values.str.isdigit()].value_counts()

no data    579
Name: count, dtype: int64

**A placeholder:** 579 cells say `no data`, a trap that wasn't counted that day. That isn't a zero, so it
can't just be replaced with 0.

## Data dictionary

| Column | What it is |
|---|---|
| `Count Date` | The day of the count, written day first (`13/05/2015`) |
| `City Wide Daily Average` | Average mosquitoes per trap across the city |
| `Trap Days` | Days of trapping this count covers (1 to 7) |
| `North West 1` to `South West 7` | One trap each, 7 per quadrant; `no data` if not counted |
| `North West Average` (and the other three) | Average per trap in that quadrant |
| `Rural AA` to `Rural II` | Traps outside the city, with no average column |